In [1]:
"""
Fair Value Calculator
----------------------
Combines bond floor and option value into a single daily fair value
for each convertible bond.

Formula:
    Fair Value (% of par) = Clean Bond Floor (% of par)
                           + Option Value (% of par)

    Mispricing            = Fair Value - Market Clean Price
                           (positive = model says bond is cheap = BUY signal)

Units — everything in % of par (market convention):
    clean_floor          already in % of par from bond_floor.py
    option_value_pct     = option_value_per_bond / par * 100
                           (column added in option_pricer.py)
    market_clean_price   already in % of par from Bloomberg

Column clarification:
    option_value         = per share value in dollars  (e.g. $4.93)
    option_value_per_bond= dollars per bond            (e.g. $42.53 = $4.93 x 8.63)
    option_value_pct     = % of par                   (e.g. 4.25% = $42.53 / $1000)

Entry signal threshold:
    mispricing > 3.0     = fair value exceeds market by $30+ per $1,000 bond

Outputs:
    data/clean/fair_value/
        {TICKER}_fair_value.csv
    data/clean/fair_value_all.csv
"""

import pandas as pd
import numpy as np
import os
import getpass
import warnings
warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────
USER = getpass.getuser()
if USER == "sarda":
    BASE_DIR = r"C:\Users\sarda\Desktop\cba2"
elif USER == "jinay":
    BASE_DIR = r"C:\Users\jinay\Desktop\cba2"
else:
    BASE_DIR = r"C:\Users\sarda\Desktop\cba2"

FLOOR_DIR   = os.path.join(BASE_DIR, "data", "clean", "bond_floors")
OPTION_DIR  = os.path.join(BASE_DIR, "data", "clean", "option_values")
OUTPUT_DIR  = os.path.join(BASE_DIR, "data", "clean", "fair_value")
os.makedirs(OUTPUT_DIR, exist_ok=True)

STATIC_PATH = os.path.join(BASE_DIR, "data", "raw", "bond_static", "CONV_BOND_DATA.xlsx")

# Entry signal threshold in % of par
# mispricing > 3.0 means model fair value exceeds market price by $30 per bond
SIGNAL_THRESHOLD = 3.0

# Tickers excluded from universe
EXCLUDE_TICKERS = {"F", "HASI", "CSWC"}


def load_static():
    """Load static bond file from sheet 'Sheet1'. Ensures 'Ticker' exists (from 'Ticker' or 'Cmn Stk Tkr')."""
    df = pd.read_excel(STATIC_PATH, sheet_name="Sheet1")
    if "Ticker" not in df.columns or df["Ticker"].isna().all():
        if "Cmn Stk Tkr" in df.columns:
            df = df.copy()
            df["Ticker"] = df["Cmn Stk Tkr"]
    return df


# ══════════════════════════════════════════════════════════════════════════
# SECTION 1 — LOAD AND MERGE ONE BOND
# ══════════════════════════════════════════════════════════════════════════

def build_fair_value(ticker, par):
    """
    Load floor and option files for one ticker, merge on date,
    and calculate fair value and mispricing.

    Merge strategy: INNER JOIN on date
        - Only dates where BOTH floor and option value exist are kept
        - Dates with floor but no option (or vice versa) are dropped
        - The row count difference (e.g. KRG 104 missing option dates)
          is handled cleanly — no NaN fair values in output

    Parameters:
        ticker : bond ticker string
        par    : face value (usually 1000)

    Returns:
        DataFrame with daily fair value and mispricing
    """
    floor_path  = os.path.join(FLOOR_DIR,  f"{ticker}_bond_floor.csv")
    option_path = os.path.join(OPTION_DIR, f"{ticker}_options.csv")

    if not os.path.exists(floor_path):
        raise FileNotFoundError(f"Floor file not found: {floor_path}")
    if not os.path.exists(option_path):
        raise FileNotFoundError(f"Option file not found: {option_path}")

    # ── Load ───────────────────────────────────────────────────────────────
    floor_df  = pd.read_csv(floor_path,  parse_dates=["date"])
    option_df = pd.read_csv(option_path, parse_dates=["date"])

    floor_rows  = len(floor_df)
    option_rows = len(option_df)

    # ── Select columns needed ──────────────────────────────────────────────
    floor_cols = ["date", "market_clean_price", "market_dirty_price",
                  "clean_floor", "dirty_floor", "accrued_interest",
                  "effective_discount_rate", "n_cashflows"]

    option_cols = ["date", "stock_price", "conversion_price",
                   "implied_vol_pct", "risk_free_rate_pct",
                   "remaining_maturity", "option_value",
                   "option_value_per_bond", "option_value_pct", "delta"]

    # Keep only columns that actually exist
    floor_cols  = [c for c in floor_cols  if c in floor_df.columns]
    option_cols = [c for c in option_cols if c in option_df.columns]

    floor_df  = floor_df[floor_cols]
    option_df = option_df[option_cols]

    # ── Ensure option_value_pct exists (backward compat with old CSVs) ────
    # option_value_pct = option_value_per_bond / par * 100  (% of par)
    if "option_value_pct" not in option_df.columns:
        if "option_value_per_bond" in option_df.columns:
            option_df["option_value_pct"] = (
                option_df["option_value_per_bond"] / par * 100
            )
            print(f"    Note: option_value_pct not in CSV — computed from "
                  f"option_value_per_bond (re-run option_pricer to fix)")
        else:
            raise ValueError(
                f"Cannot compute option_value_pct for {ticker} — "
                "neither option_value_pct nor option_value_per_bond found"
            )

    # ── Inner merge on date ────────────────────────────────────────────────
    merged = pd.merge(floor_df, option_df, on="date", how="inner")
    merged = merged.sort_values("date").reset_index(drop=True)

    date_diff = abs(floor_rows - option_rows)
    if date_diff > 0:
        print(f"    Date alignment: floor={floor_rows}, "
              f"option={option_rows}, matched={len(merged)} "
              f"(dropped {date_diff} unmatched dates)")

    # ── Fair value ─────────────────────────────────────────────────────────
    # clean_floor      = % of par  (e.g. 90.00)
    # option_value_pct = % of par  (e.g.  4.25)
    # fair_value_clean = % of par  (e.g. 94.25)
    merged["fair_value_clean"] = (
        merged["clean_floor"] + merged["option_value_pct"]
    )

    # Dirty fair value = clean + accrued interest
    merged["fair_value_dirty"] = (
        merged["fair_value_clean"] + merged["accrued_interest"]
    )

    # ── Mispricing ─────────────────────────────────────────────────────────
    # Positive = model fair value > market = bond is CHEAP = BUY signal
    # Negative = model fair value < market = bond is RICH
    merged["mispricing_pct"] = (
        merged["fair_value_clean"] - merged["market_clean_price"]
    )

    # Dollar mispricing per bond (% of par converted to dollars)
    merged["mispricing_usd"] = merged["mispricing_pct"] / 100 * par

    # ── Signal flag ────────────────────────────────────────────────────────
    merged["signal"] = merged["mispricing_pct"] > SIGNAL_THRESHOLD

    # ── Add ticker ─────────────────────────────────────────────────────────
    merged.insert(0, "ticker", ticker)

    return merged


# ══════════════════════════════════════════════════════════════════════════
# SECTION 2 — RUN ALL BONDS
# ══════════════════════════════════════════════════════════════════════════

def run_all_bonds():
    print("=" * 65)
    print("  FAIR VALUE CALCULATOR")
    print("  Fair Value = Clean Bond Floor + Option Value (% of par)")
    print(f"  Entry signal threshold: mispricing > {SIGNAL_THRESHOLD}% of par")
    print("=" * 65)

    static_df = load_static()
    tickers   = [
        t for t in static_df["Ticker"].dropna().unique().tolist()
        if t not in EXCLUDE_TICKERS
    ]

    if not tickers:
        sample = static_df["Ticker"].head(5).tolist()
        print(f"\nFound 0 bonds. Ticker column sample: {sample}")
        print("  Check CONV_BOND_DATA.xlsx sheet 'Sheet1': ensure 'Ticker' or 'Cmn Stk Tkr' has ticker symbols.")
        return []
    print(f"\nBonds: {len(tickers)} — {tickers}\n")

    all_results = []

    for i, ticker in enumerate(tickers, 1):
        print(f"[{i}/{len(tickers)}] {ticker}...")

        try:
            par_row = static_df[static_df["Ticker"] == ticker]["Par Amt"]
            par     = float(par_row.values[0]) if len(par_row) > 0 else 1000.0

            result_df = build_fair_value(ticker, par)

            if result_df.empty:
                print(f"    WARNING: No results for {ticker}\n")
                continue

            # ── Summary ────────────────────────────────────────────────────
            n_signal       = result_df["signal"].sum()
            pct_signal     = n_signal / len(result_df) * 100
            avg_misp_pct   = result_df["mispricing_pct"].mean()
            avg_misp_usd   = result_df["mispricing_usd"].mean()

            print(f"    Rows: {len(result_df)} | "
                  f"{result_df['date'].min().date()} to "
                  f"{result_df['date'].max().date()}")
            print(f"    Fair value range:   "
                  f"{result_df['fair_value_clean'].min():.2f} – "
                  f"{result_df['fair_value_clean'].max():.2f}")
            print(f"    Market price range: "
                  f"{result_df['market_clean_price'].min():.2f} – "
                  f"{result_df['market_clean_price'].max():.2f}")
            print(f"    Mispricing range:   "
                  f"{result_df['mispricing_pct'].min():.2f}% to "
                  f"{result_df['mispricing_pct'].max():.2f}%  "
                  f"(avg: {avg_misp_pct:.2f}% = ${avg_misp_usd:.1f}/bond)")
            print(f"    Signal days: {n_signal}/{len(result_df)} "
                  f"({pct_signal:.1f}%) where mispricing > {SIGNAL_THRESHOLD}%")
            print()

            # ── Save ───────────────────────────────────────────────────────
            out_path = os.path.join(OUTPUT_DIR, f"{ticker}_fair_value.csv")
            result_df.to_csv(out_path, index=False)

            all_results.append(result_df)

        except FileNotFoundError as e:
            print(f"    SKIPPED — {e}\n")
        except Exception as e:
            import traceback
            print(f"    ERROR — {ticker}: {e}")
            traceback.print_exc()
            print()

    # ── Save combined file ─────────────────────────────────────────────────
    if all_results:
        combined      = pd.concat(all_results, ignore_index=True)
        combined_path = os.path.join(BASE_DIR, "data", "clean", "fair_value_all.csv")
        combined.to_csv(combined_path, index=False)

        print("=" * 65)
        print(f"Combined file saved → data/clean/fair_value_all.csv")
        print(f"Total rows: {len(combined)} across {len(all_results)} bonds")

        # ── Cross-bond summary table ───────────────────────────────────────
        print(f"\nCross-bond mispricing summary:")
        print(f"  {'Ticker':<8} {'Rows':>6} {'Avg Misp%':>10} "
              f"{'Avg Misp USD':>13} {'Signal Days':>12} {'Signal%':>8}")
        print(f"  {'-'*62}")
        for df in all_results:
            t       = df["ticker"].iloc[0]
            n       = len(df)
            am_pct  = df["mispricing_pct"].mean()
            am_usd  = df["mispricing_usd"].mean()
            sd      = df["signal"].sum()
            sp      = sd / n * 100
            print(f"  {t:<8} {n:>6} {am_pct:>9.2f}% "
                  f"${am_usd:>11.1f} {sd:>12} {sp:>7.1f}%")

        # ── Recommended bonds for backtest (better portfolio) ─────────────────
        MIN_ROWS = 250
        SIGNAL_PCT_LO, SIGNAL_PCT_HI = 15.0, 99.0   # exclude only no-signal; include high-signal bonds (100% = persistently cheap)
        AVG_MISP_LO, AVG_MISP_HI = 1.5, 16.0       # meaningful mispricing, not extreme
        summary = []
        for df in all_results:
            t = df["ticker"].iloc[0]
            n = len(df)
            am_pct = df["mispricing_pct"].mean()
            sp = df["signal"].sum() / n * 100
            summary.append({"ticker": t, "rows": n, "avg_misp_pct": am_pct, "signal_pct": sp})
        summary_df = pd.DataFrame(summary)
        rec = summary_df[
            (summary_df["rows"] >= MIN_ROWS) &
            (summary_df["signal_pct"] >= SIGNAL_PCT_LO) & (summary_df["signal_pct"] <= SIGNAL_PCT_HI) &
            (summary_df["avg_misp_pct"] >= AVG_MISP_LO) & (summary_df["avg_misp_pct"] <= AVG_MISP_HI)
        ]
        rec_tickers = rec["ticker"].tolist()
        rec_path = os.path.join(BASE_DIR, "data", "clean", "backtest_recommended_tickers.csv")
        os.makedirs(os.path.dirname(rec_path), exist_ok=True)
        pd.DataFrame({"ticker": rec_tickers}).to_csv(rec_path, index=False)
        print(f"\nRecommended for backtest: {len(rec_tickers)} bonds "
              f"(rows>={MIN_ROWS}, signal {SIGNAL_PCT_LO}-{SIGNAL_PCT_HI}%, avg misp {AVG_MISP_LO}-{AVG_MISP_HI}%)")
        print(f"  → {rec_tickers}")
        print(f"  Saved → data/clean/backtest_recommended_tickers.csv")

    return all_results


# ══════════════════════════════════════════════════════════════════════════
# SECTION 3 — TEST SINGLE BOND
# ══════════════════════════════════════════════════════════════════════════

def test_single_bond(ticker="MTH"):
    print("=" * 65)
    print(f"  FAIR VALUE TEST — {ticker}")
    print("=" * 65)

    static_df = load_static()
    par_row   = static_df[static_df["Ticker"] == ticker]["Par Amt"]
    par       = float(par_row.values[0]) if len(par_row) > 0 else 1000.0

    df = build_fair_value(ticker, par)

    print(f"\nFirst 5 rows:")
    print(df[["date", "stock_price", "clean_floor", "option_value_pct",
              "fair_value_clean", "market_clean_price",
              "mispricing_pct", "mispricing_usd", "signal"]].head().to_string())

    print(f"\nLast 5 rows:")
    print(df[["date", "stock_price", "clean_floor", "option_value_pct",
              "fair_value_clean", "market_clean_price",
              "mispricing_pct", "mispricing_usd", "signal"]].tail().to_string())

    print(f"\nSummary statistics:")
    print(df[["clean_floor", "option_value_pct", "fair_value_clean",
              "market_clean_price", "mispricing_pct",
              "mispricing_usd"]].describe().round(4))

    # ── Sanity checks ──────────────────────────────────────────────────────
    print(f"\nSanity checks:")

    # Fair value must always be above bond floor (option value is always >= 0)
    fv_below_floor = (df["fair_value_clean"] < df["clean_floor"]).sum()
    print(f"  Days where fair value < floor (should be 0): {fv_below_floor}")

    # Mispricing distribution
    pct_cheap = (df["mispricing_pct"] > 0).sum() / len(df) * 100
    pct_rich  = (df["mispricing_pct"] < 0).sum() / len(df) * 100
    print(f"  Days bond looks cheap (mispricing > 0): {pct_cheap:.1f}%")
    print(f"  Days bond looks rich  (mispricing < 0): {pct_rich:.1f}%")

    # Signal days
    signal_days = df["signal"].sum()
    print(f"  Signal days (mispricing > {SIGNAL_THRESHOLD}%): "
          f"{signal_days} ({signal_days/len(df)*100:.1f}%)")

    # Full decomposition on most recent date
    latest = df.iloc[-1]
    print(f"\n  Decomposition on {latest['date'].date()}:")
    print(f"    Stock price:              ${latest['stock_price']:.2f}")
    print(f"    Bond floor (% of par):    {latest['clean_floor']:.4f}% "
          f"= ${latest['clean_floor']/100*par:.2f}")
    print(f"    Option value (% of par):  {latest['option_value_pct']:.4f}% "
          f"= ${latest['option_value_pct']/100*par:.2f}")
    print(f"    Fair value (% of par):    {latest['fair_value_clean']:.4f}% "
          f"= ${latest['fair_value_clean']/100*par:.2f}")
    print(f"    Market price (% of par):  {latest['market_clean_price']:.4f}% "
          f"= ${latest['market_clean_price']/100*par:.2f}")
    print(f"    Mispricing:               {latest['mispricing_pct']:.4f}% "
          f"= ${latest['mispricing_usd']:.2f}/bond")
    print(f"    Signal:                   {latest['signal']}")

    return df


# ══════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    import sys

    if len(sys.argv) > 1 and sys.argv[1] == "test":
        ticker = sys.argv[2] if len(sys.argv) > 2 else "MTH"
        test_single_bond(ticker)
    else:
        run_all_bonds()

  FAIR VALUE CALCULATOR
  Fair Value = Clean Bond Floor + Option Value (% of par)
  Entry signal threshold: mispricing > 3.0% of par

Bonds: 19 — ['KRG', 'PPL1', 'EXC', 'BXP', 'EEFT', 'WEC1', 'DLR', 'WEC2', 'WEC3', 'REXR1', 'REXR2', 'MTH', 'NEE', 'FRT', 'EVRG', 'CDP', 'DUK', 'VTR', 'PPL2']

[1/19] KRG...
    Date alignment: floor=1288, option=1184, matched=1178 (dropped 104 unmatched dates)
    Rows: 1178 | 2021-03-22 to 2026-02-24
    Fair value range:   94.39 – 138.88
    Market price range: 84.51 – 115.81
    Mispricing range:   -0.05% to 41.41%  (avg: 13.68% = $136.8/bond)
    Signal days: 1117/1178 (94.8%) where mispricing > 3.0%

[2/19] PPL1...
    Date alignment: floor=53, option=68, matched=51 (dropped 15 unmatched dates)
    Rows: 51 | 2025-11-24 to 2026-02-24
    Fair value range:   104.15 – 112.45
    Market price range: 98.88 – 104.06
    Mispricing range:   4.44% to 8.55%  (avg: 7.08% = $70.8/bond)
    Signal days: 51/51 (100.0%) where mispricing > 3.0%

[3/19] EXC...
    

In [2]:
"""
Bloomberg CV Sanity Check
--------------------------
Compares our model's option value and delta on the most recent
available date against Bloomberg's snapshot values:
    CV Equity Option Value  → Bloomberg's per-share option value
    CV Model Delta S %      → Bloomberg's delta

This validates whether our CRR binomial tree is calibrated correctly.

What we expect:
    - Option value within ~30% of Bloomberg's figure
    - Delta within ~0.10 of Bloomberg's figure
    - Systematic over/under-estimation points to a model input issue

Most common causes of large discrepancies:
    1. Implied volatility too high → option value too high
    2. Wrong conversion price or ratio
    3. Risk-free rate mismatch
    4. Bloomberg uses call provisions which we ignore

Output:
    Printed comparison table for all bonds
    data/clean/bloomberg_sanity_check.csv
"""

import pandas as pd
import numpy as np
import os
import getpass
import warnings
warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────
USER = getpass.getuser()
if USER == "sarda":
    BASE_DIR = r"C:\Users\sarda\Desktop\cba2"
elif USER == "jinay":
    BASE_DIR = r"C:\Users\jinay\Desktop\cba2"
else:
    BASE_DIR = r"C:\Users\sarda\Desktop\cba2"

STATIC_PATH = os.path.join(BASE_DIR, "data", "raw",   "bond_static", "CONV_BOND_DATA.xlsx")
OPTION_DIR  = os.path.join(BASE_DIR, "data", "clean", "option_values")
CLEAN_DIR   = os.path.join(BASE_DIR, "data", "clean")

EXCLUDE_TICKERS = {"F", "HASI", "CSWC"}


def load_static():
    """Load static bond file from sheet 'Sheet1'. Ensures 'Ticker' exists (from 'Ticker' or 'Cmn Stk Tkr')."""
    df = pd.read_excel(STATIC_PATH, sheet_name="Sheet1")
    if "Ticker" not in df.columns or df["Ticker"].isna().all():
        if "Cmn Stk Tkr" in df.columns:
            df = df.copy()
            df["Ticker"] = df["Cmn Stk Tkr"]
    return df


# ══════════════════════════════════════════════════════════════════════════
# SECTION 1 — RUN COMPARISON
# ══════════════════════════════════════════════════════════════════════════

def run_bloomberg_sanity_check():
    print("=" * 75)
    print("  BLOOMBERG CV SANITY CHECK")
    print("  Comparing model output vs Bloomberg CBBT snapshot")
    print("=" * 75)

    static_df = load_static()
    tickers   = [
        t for t in static_df["Ticker"].dropna().unique().tolist()
        if t not in EXCLUDE_TICKERS
    ]

    if not tickers:
        sample = static_df["Ticker"].head(5).tolist()
        print(f"\nFound 0 bonds. Ticker column sample: {sample}")
        print("  Check CONV_BOND_DATA.xlsx sheet 'Sheet1': ensure 'Ticker' or 'Cmn Stk Tkr' has ticker symbols.")
        return

    results = []

    for ticker in tickers:
        static_row = static_df[static_df["Ticker"] == ticker]

        # ── Bloomberg snapshot values ──────────────────────────────────────
        bbg_option_per_share = float(static_row["CV Equity Option Value"].values[0])
        bbg_delta_pct        = float(static_row["CV Model Delta S %"].values[0])
        bbg_delta            = bbg_delta_pct / 100
        conversion_ratio     = float(static_row["Conversion Ratio"].values[0])
        conversion_price     = float(static_row["CV Conversion Price"].values[0])
        par                  = float(static_row["Par Amt"].values[0])

        # Bloomberg option value per bond = per share * conversion ratio
        bbg_option_per_bond  = bbg_option_per_share * conversion_ratio
        bbg_option_pct       = bbg_option_per_bond / par * 100

        # ── Load our model's most recent date ──────────────────────────────
        option_path = os.path.join(OPTION_DIR, f"{ticker}_options.csv")
        if not os.path.exists(option_path):
            print(f"  {ticker}: option file not found — skipped")
            continue

        opt_df  = pd.read_csv(option_path, parse_dates=["date"])
        opt_df  = opt_df.sort_values("date")
        latest  = opt_df.iloc[-1]

        our_option_per_share = float(latest["option_value"])
        our_delta            = float(latest["delta"])
        our_option_per_bond  = float(latest["option_value_per_bond"])
        our_option_pct       = our_option_per_bond / par * 100
        our_iv               = float(latest["implied_vol_pct"])
        our_rf               = float(latest["risk_free_rate_pct"])
        our_T                = float(latest["remaining_maturity"])
        our_S                = float(latest["stock_price"])
        model_date           = latest["date"]

        # ── Differences ────────────────────────────────────────────────────
        option_diff_pct = ((our_option_per_share - bbg_option_per_share)
                           / bbg_option_per_share * 100
                           if bbg_option_per_share != 0 else np.nan)
        delta_diff      = our_delta - bbg_delta

        # ── Flag ───────────────────────────────────────────────────────────
        # Flag if option value differs by more than 30% OR delta differs by > 0.15
        option_flag = abs(option_diff_pct) > 30 if not np.isnan(option_diff_pct) else True
        delta_flag  = abs(delta_diff) > 0.15

        if option_flag and delta_flag:
            flag = "⚠️  BOTH"
        elif option_flag:
            flag = "⚠️  OPTION"
        elif delta_flag:
            flag = "⚠️  DELTA"
        else:
            flag = "✅ OK"

        results.append({
            "ticker"              : ticker,
            "model_date"          : model_date,
            "stock_price"         : round(our_S, 2),
            "conversion_price"    : round(conversion_price, 2),
            "conversion_ratio"    : round(conversion_ratio, 4),
            "remaining_maturity"  : round(our_T, 3),
            "implied_vol_pct"     : round(our_iv, 2),
            "risk_free_rate_pct"  : round(our_rf, 3),
            # Option value per share
            "bbg_option_per_share": round(bbg_option_per_share, 4),
            "our_option_per_share": round(our_option_per_share, 4),
            "option_diff_pct"     : round(option_diff_pct, 1) if not np.isnan(option_diff_pct) else np.nan,
            # Option value per bond (% of par)
            "bbg_option_pct"      : round(bbg_option_pct, 4),
            "our_option_pct"      : round(our_option_pct, 4),
            # Delta
            "bbg_delta"           : round(bbg_delta, 4),
            "our_delta"           : round(our_delta, 4),
            "delta_diff"          : round(delta_diff, 4),
            # Flag
            "flag"                : flag,
        })

    # ── Print results ──────────────────────────────────────────────────────
    df = pd.DataFrame(results)

    print(f"\n{'Ticker':<6} {'Date':<12} {'S':>7} {'K':>7} {'T':>5} "
          f"{'IV%':>6} {'RF%':>5} | "
          f"{'BBG Opt':>8} {'Our Opt':>8} {'Diff%':>7} | "
          f"{'BBG Dlt':>8} {'Our Dlt':>8} {'DltDiff':>8} | "
          f"{'Flag':<12}")
    print("-" * 120)

    for _, row in df.iterrows():
        print(
            f"{row['ticker']:<6} "
            f"{str(row['model_date'])[:10]:<12} "
            f"{row['stock_price']:>7.2f} "
            f"{row['conversion_price']:>7.2f} "
            f"{row['remaining_maturity']:>5.2f} "
            f"{row['implied_vol_pct']:>6.1f} "
            f"{row['risk_free_rate_pct']:>5.2f} | "
            f"{row['bbg_option_per_share']:>8.3f} "
            f"{row['our_option_per_share']:>8.3f} "
            f"{row['option_diff_pct']:>6.1f}% | "
            f"{row['bbg_delta']:>8.4f} "
            f"{row['our_delta']:>8.4f} "
            f"{row['delta_diff']:>+8.4f} | "
            f"{row['flag']}"
        )

    # ── Summary ────────────────────────────────────────────────────────────
    n_ok      = (df["flag"] == "✅ OK").sum()
    n_flag    = len(df) - n_ok
    avg_diff  = df["option_diff_pct"].abs().mean()

    print(f"\n{'='*75}")
    print(f"  Summary: {n_ok}/{len(df)} bonds within tolerance | "
          f"{n_flag} flagged | Avg option diff: {avg_diff:.1f}%")

    # ── Diagnose systematic bias ───────────────────────────────────────────
    avg_signed_diff = df["option_diff_pct"].mean()
    if avg_signed_diff > 20:
        print(f"\n  DIAGNOSIS: Model is systematically OVERVALUING options")
        print(f"  Average overvaluation: {avg_signed_diff:.1f}%")
        print(f"  Most likely cause: Implied volatility inputs are too high")
        print(f"  Average IV used: {df['implied_vol_pct'].mean():.1f}%")
        print(f"  Check: Is the IV column in your equity files already in %?")
        print(f"  If Bloomberg gives IV as 35.08 and you are NOT dividing by 100")
        print(f"  before passing to the tree, sigma=35.08 instead of 0.3508")
        print(f"  That would massively inflate option values.")
    elif avg_signed_diff < -20:
        print(f"\n  DIAGNOSIS: Model is systematically UNDERVALUING options")
        print(f"  Average undervaluation: {abs(avg_signed_diff):.1f}%")
        print(f"  Most likely cause: IV too low, or wrong conversion parameters")
    else:
        print(f"\n  DIAGNOSIS: No systematic bias detected")
        print(f"  Average signed difference: {avg_signed_diff:.1f}%")

    # ── Per-bond detail for flagged bonds ──────────────────────────────────
    flagged = df[df["flag"] != "✅ OK"]
    if len(flagged) > 0:
        print(f"\n  Flagged bonds — detailed breakdown:")
        for _, row in flagged.iterrows():
            moneyness = row["stock_price"] / row["conversion_price"] * 100
            print(f"\n  {row['ticker']}:")
            print(f"    Moneyness (S/K):       {moneyness:.1f}%  "
                  f"({'ITM' if moneyness > 100 else 'OTM'})")
            print(f"    Remaining maturity:    {row['remaining_maturity']:.2f} years")
            print(f"    Implied vol:           {row['implied_vol_pct']:.1f}%")
            print(f"    Risk-free rate:        {row['risk_free_rate_pct']:.2f}%")
            print(f"    BBG option per share:  ${row['bbg_option_per_share']:.3f}")
            print(f"    Our option per share:  ${row['our_option_per_share']:.3f}  "
                  f"({row['option_diff_pct']:+.1f}% vs BBG)")
            print(f"    BBG option (% of par): {row['bbg_option_pct']:.3f}%")
            print(f"    Our option (% of par): {row['our_option_pct']:.3f}%")
            print(f"    BBG delta:             {row['bbg_delta']:.4f}")
            print(f"    Our delta:             {row['our_delta']:.4f}  "
                  f"(diff: {row['delta_diff']:+.4f})")

    # ── Save ───────────────────────────────────────────────────────────────
    out_path = os.path.join(CLEAN_DIR, "bloomberg_sanity_check.csv")
    df.to_csv(out_path, index=False)
    print(f"\n  Full results saved → data/clean/bloomberg_sanity_check.csv")
    print("=" * 75)

    return df


# ══════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    run_bloomberg_sanity_check()

  BLOOMBERG CV SANITY CHECK
  Comparing model output vs Bloomberg CBBT snapshot

Ticker Date               S       K     T    IV%   RF% |  BBG Opt  Our Opt   Diff% |  BBG Dlt  Our Dlt  DltDiff | Flag        
------------------------------------------------------------------------------------------------------------------------
KRG    2026-03-04     25.90   23.61  1.08   28.3  3.56 |   33.032    4.707  -85.7% |   0.8539   0.7228  -0.1311 | ⚠️  OPTION
PPL1   2026-03-04     38.59   42.66  4.75   19.8  3.52 |   11.857    7.702  -35.0% |   0.5482   0.6435  +0.0953 | ⚠️  OPTION
EXC    2026-03-04     49.25   57.11  3.03   21.2  3.40 |    8.040    6.271  -22.0% |   0.4575   0.5236  +0.0661 | ✅ OK
BXP    2026-03-04     56.36   92.44  4.58   32.5  3.50 |    4.036    9.162  127.0% |   0.3225   0.4462  +0.1237 | ⚠️  OPTION
EEFT   2026-03-04     74.33  127.04  4.58   39.3  3.50 |   14.123   15.520    9.9% |   0.5429   0.4890  -0.0539 | ✅ OK
WEC1   2026-03-04    117.55  128.37  2.25   18.6  3.39 |  